In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ===========================================
# BERT(address(in).csv with last_product_ids) vs XGB(test_predictions_orders_threshold_*.csv)
# Instacart 방식(주문 단위 F1, 빈 예측=‘None’) 성능 비교 파이프라인
# ===========================================
import os, json, re, time
import numpy as np
import pandas as pd

# ---------- 0) 경로 설정 ----------
DATA_DIR = "/content/drive/MyDrive/data/instacart"
MASTER_PATH = os.path.join(DATA_DIR, "master_dataset_with_roles_final.csv")

# 팀원 BERT 결과 (확인: cols=['user_id','eval_set','last_product_ids'])
BERT_PATH = os.path.join(DATA_DIR, "address(in).csv")

# 내 XGB 예측(주문별 products)
XGB_ORD_PATH = os.path.join(DATA_DIR, "test_predictions_orders_threshold_0.32.csv")

# (옵션) XGB 상세 확률 파일이 있으면 AUC/Logloss도 계산
XGB_DETAIL_PATH = os.path.join(DATA_DIR, "test_predictions_detailed.csv")

# 출력 경로
OUT_SUMMARY_CSV  = os.path.join(DATA_DIR, "bert_xgb_comparison_summary.csv")
OUT_SUMMARY_JSON = os.path.join(DATA_DIR, "bert_xgb_comparison_summary.json")
OUT_PER_ORDER    = os.path.join(DATA_DIR, "bert_xgb_per_order_f1.csv")
OUT_SAMPLES_CSV  = os.path.join(DATA_DIR, "bert_xgb_samples_check.csv")    # 파일들 헤더/샘플 확인용
OUT_BERT_TOP50   = os.path.join(DATA_DIR, "bert_better_top50.csv")
OUT_XGB_TOP50    = os.path.join(DATA_DIR, "xgb_better_top50.csv")

# ---------- 1) 유틸 ----------
def f1_single(true_set: set, pred_set: set) -> float:
    # Instacart rule: 둘 다 빈 집합이면 'None' 매칭으로 F1=1
    if len(true_set) == 0 and len(pred_set) == 0:
        return 1.0
    if len(pred_set) == 0:
        pred_set = {"None"}
    if len(true_set) == 0:
        true_set = {"None"}
    tp = len(true_set & pred_set)
    if tp == 0:
        return 0.0
    precision = tp / len(pred_set)
    recall    = tp / len(true_set)
    return 0.0 if (precision+recall)==0 else 2*precision*recall/(precision+recall)

def parse_products_to_list(s) -> list:
    """문자열에서 product_id 리스트 파싱: '1 2', '1, 2', '[1,2]' 등 모두 허용."""
    if pd.isna(s):
        return []
    s = str(s).strip()
    if s == "" or s.lower() == "none":
        return []
    # 숫자 토큰 추출(콤마/괄호/공백 등 모두 무시)
    toks = re.findall(r"\d+", s)
    if len(toks) > 0:
        return toks
    # 혹시 숫자 토큰이 아니면 공백 분리
    return [t for t in s.split() if t]

def first_existing(cands, cols):
    for c in cands:
        if c in cols: return c
    return None

# ---------- 2) GT(정답) 구성: eval_set=='train'(6번) ----------
print("[INFO] Load master:", MASTER_PATH)
df_all = pd.read_csv(MASTER_PATH, low_memory=False)

# 7번(test) 제거
if "eval_set" in df_all.columns:
    ev = df_all["eval_set"].astype(str).str.lower()
    df_all = df_all.loc[~ev.eq("test")].copy()
    mask_test = df_all["eval_set"].astype(str).str.lower().eq("train")
else:
    # fallback: role_test 컬럼
    mask_test = (df_all["role_test"].fillna(0).astype(int)==1) if "role_test" in df_all.columns else pd.Series([False]*len(df_all))

cols = df_all.columns.tolist()
target_col  = first_existing(['reordered','label','target','y','is_reordered'], cols)
product_col = first_existing(['product_id','pid','product'], cols)
order_col   = first_existing(['order_id','oid','order'], cols)
member_col  = first_existing(['user_id','member_id','uid','user'], cols)
assert target_col and product_col and (order_col or member_col), "필수 컬럼 누락: target/product/(order or user)."

order_key = order_col if order_col else member_col

df_test = df_all.loc[mask_test, [order_key, product_col, target_col] + ([member_col] if (member_col and member_col!=order_key) else [])].copy()
# 이진화
if set(pd.unique(df_test[target_col].dropna())) - {0,1}:
    df_test[target_col] = (df_test[target_col] > 0).astype(np.int8)

# GT: 주문→정답 집합(문자열화)
df_gt = df_test[[order_key, product_col, target_col]].copy()
df_gt[order_key]   = df_gt[order_key].astype(str)
df_gt[product_col] = df_gt[product_col].astype(str)
true_sets = df_gt.loc[df_gt[target_col]==1].groupby(order_key)[product_col].agg(lambda s: set(s.tolist())).to_dict()

# user_id → (train) order_id 매핑 (BERT가 user_id만 줄 경우 대비)
user_to_order = None
if (member_col is not None) and (order_col is not None):
    tmp_map = df_test[[member_col, order_col]].drop_duplicates()
    tmp_map[member_col] = tmp_map[member_col].astype(str)
    tmp_map[order_col]  = tmp_map[order_col].astype(str)
    user_to_order = dict(zip(tmp_map[member_col], tmp_map[order_col]))

print(f"[INFO] GT orders: {len(true_sets):,}  | rows in df_test: {len(df_test):,}")

# ---------- 3) XGB 예측(주문별 products) 로드 ----------
print("[INFO] Load XGB orders:", XGB_ORD_PATH)
xgb_ord = pd.read_csv(XGB_ORD_PATH)
id_order_cands   = ["order_id","oid","order","test_order_id", order_key]
products_cands   = ["products","predicted_products","prediction","preds"]

xgb_id_col  = first_existing(id_order_cands, xgb_ord.columns.tolist())
xgb_prodcol = first_existing(products_cands, xgb_ord.columns.tolist())
assert xgb_id_col and xgb_prodcol, "XGB 예측 파일에서 주문ID 또는 products 컬럼을 찾지 못했습니다."

# member만 있으면 order로 변환
if (xgb_id_col == member_col) and (user_to_order is not None):
    xgb_ord[xgb_id_col] = xgb_ord[xgb_id_col].astype(str).map(user_to_order)

# dict화
xgb_pred_sets = {}
for oid, prod_str in xgb_ord[[xgb_id_col, xgb_prodcol]].itertuples(index=False, name=None):
    oid = str(oid)
    xgb_pred_sets[oid] = set(parse_products_to_list(prod_str))

print(f"[INFO] XGB orders loaded: {len(xgb_pred_sets):,}")

# ---------- 4) BERT 예측(address(in).csv) 로드 ----------
print("[INFO] Load BERT file:", BERT_PATH)
# 인코딩 이슈 방지
try:
    bert = pd.read_csv(BERT_PATH)
except UnicodeDecodeError:
    bert = pd.read_csv(BERT_PATH, encoding="utf-8-sig")

# (분석 결과 반영) 컬럼: user_id, eval_set, last_product_ids
bert_cols = bert.columns.tolist()
print("[INFO] BERT cols:", bert_cols)

# eval_set == 'train'만 사용
if "eval_set" in bert_cols:
    bert = bert[bert["eval_set"].astype(str).str.lower() == "train"].copy()

# id/product 컬럼 확인
id_member_cands  = ["user_id","member_id","uid","user"]
bert_id_col = first_existing(["order_id","oid","order"] + id_member_cands, bert.columns.tolist())

# 제품 문자열 컬럼: 'last_product_ids'를 우선 사용 (없으면 products 후보)
bert_prodcol = "last_product_ids" if "last_product_ids" in bert.columns else first_existing(products_cands, bert.columns.tolist())

assert bert_id_col is not None, "BERT 파일에서 주문ID/유저ID 컬럼을 찾지 못했습니다."
assert bert_prodcol is not None, "BERT 파일에서 제품 리스트 컬럼(last_product_ids/products)이 없습니다."

# id가 user면 (train)order로 치환
if (bert_id_col in id_member_cands) and (user_to_order is not None):
    before_n = len(bert)
    bert[bert_id_col] = bert[bert_id_col].astype(str).map(user_to_order)
    # 매핑 실패 제거
    miss = bert[bert_id_col].isna().sum()
    if miss > 0:
        print(f"[WARN] BERT user→order 매핑 실패 {miss:,}건 제외 (before {before_n:,} → after {(before_n-miss):,})")
    bert = bert.dropna(subset=[bert_id_col])

# dict화
bert_pred_sets = {}
for oid, prod_str in bert[[bert_id_col, bert_prodcol]].itertuples(index=False, name=None):
    if pd.isna(oid):
        continue
    oid = str(oid)
    bert_pred_sets[oid] = set(parse_products_to_list(prod_str))

print(f"[INFO] BERT orders loaded: {len(bert_pred_sets):,}")

# ---------- 5) 성능 계산 (Instacart 주문 단위 F1) ----------
orders_common_xgb  = set(xgb_pred_sets.keys())  & set(true_sets.keys())
orders_common_bert = set(bert_pred_sets.keys()) & set(true_sets.keys())

def compute_macro_f1(orders_common, pred_sets, true_sets):
    if len(orders_common) == 0:
        return 0.0, []
    f1s = []
    for oid in orders_common:
        f1s.append(f1_single(true_sets[oid], pred_sets.get(oid, set())))
    return float(np.mean(f1s)), f1s

f1_xgb,  f1_list_xgb  = compute_macro_f1(orders_common_xgb,  xgb_pred_sets,  true_sets)
f1_bert, f1_list_bert = compute_macro_f1(orders_common_bert, bert_pred_sets, true_sets)

coverage_xgb  = len(orders_common_xgb)  / max(1, len(true_sets))
coverage_bert = len(orders_common_bert) / max(1, len(true_sets))

print("\n===== 성능 요약 (Instacart F1) =====")
print(f"XGB  | orders used: {len(orders_common_xgb):,} / {len(true_sets):,} (cov={coverage_xgb:.2%}) | F1 = {f1_xgb:.5f}")
print(f"BERT | orders used: {len(orders_common_bert):,} / {len(true_sets):,} (cov={coverage_bert:.2%}) | F1 = {f1_bert:.5f}")

# ---------- 6) 주문별 F1 비교 테이블 ----------
orders_both = list(orders_common_xgb & orders_common_bert)
per_order_rows = []
for oid in orders_both:
    ts = true_sets[oid]
    xs = xgb_pred_sets.get(oid, set())
    bs = bert_pred_sets.get(oid, set())
    per_order_rows.append({
        "order_id": oid,
        "F1_XGB": f1_single(ts, xs),
        "F1_BERT": f1_single(ts, bs),
        "F1_diff(BERT-XGB)": f1_single(ts, bs) - f1_single(ts, xs),
        "True_size": len(ts),
        "XGB_pred_size": len(xs),
        "BERT_pred_size": len(bs)
    })
per_order_df = pd.DataFrame(per_order_rows)

# ---------- 7) (옵션) XGB AUC/Logloss: 상세 확률이 있으면 계산 ----------
def safe_log_loss(y_true, y_pred):
    p = np.clip(np.asarray(y_pred, dtype=np.float64), 1e-15, 1-1e-15)
    from sklearn.metrics import log_loss
    return float(log_loss(y_true, p))

def compute_auc_logloss_from_detail(detail_path, order_key, product_col, target_col, proba_col="proba_ens", max_rows=2_000_000, seed=2025):
    if not os.path.exists(detail_path):
        return None, None
    d = pd.read_csv(detail_path)
    need = [order_key, product_col, target_col, proba_col]
    if not all(c in d.columns for c in need):
        return None, None
    # 샘플링(속도)
    if len(d) > max_rows:
        d = d.sample(n=max_rows, random_state=seed)
        print(f"[metrics] sampled {len(d):,} rows for AUC/Logloss from {os.path.basename(detail_path)}")
    y = d[target_col].values
    p = np.clip(d[proba_col].values.astype(np.float64), 1e-15, 1-1e-15)
    from sklearn.metrics import roc_auc_score
    auc = float(roc_auc_score(y, p)) if len(np.unique(y))>1 else np.nan
    ll  = safe_log_loss(y, p)
    return auc, ll

AUC_XGB, LL_XGB = compute_auc_logloss_from_detail(XGB_DETAIL_PATH, order_key, product_col, target_col, proba_col="proba_ens")

# ---------- 8) 샘플/요약/저장 ----------
# 샘플 헤더 저장(점검용)
sample_out = []
for name, df_src in [("xgb", xgb_ord.head(5)), ("bert", bert.head(5))]:
    tmp = df_src.copy()
    tmp.insert(0, "source", name)
    sample_out.append(tmp)
pd.concat(sample_out, ignore_index=True, sort=False).to_csv(OUT_SAMPLES_CSV, index=False)

summary_rows = [
    {"model":"XGB_orders",  "macro_F1":f1_xgb,  "orders_used":len(orders_common_xgb),  "coverage":coverage_xgb,  "AUC":AUC_XGB, "Logloss":LL_XGB},
    {"model":"BERT_orders", "macro_F1":f1_bert, "orders_used":len(orders_common_bert), "coverage":coverage_bert, "AUC":None,   "Logloss":None}
]
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUT_SUMMARY_CSV, index=False)

with open(OUT_SUMMARY_JSON, "w") as f:
    json.dump({
        "summary": summary_rows,
        "notes": {
            "gt_orders": len(true_sets),
            "orders_common_both": len(orders_both),
            "xgb_id_col": xgb_id_col,
            "xgb_products_col": xgb_prodcol,
            "bert_id_col": bert_id_col,
            "bert_products_col": bert_prodcol,
            "filtered_bert_on_eval_set_train": "yes" if "eval_set" in bert_cols else "no_eval_set_col",
            "master_path": MASTER_PATH,
            "xgb_orders_path": XGB_ORD_PATH,
            "bert_path": BERT_PATH,
            "xgb_detail_path": XGB_DETAIL_PATH if os.path.exists(XGB_DETAIL_PATH) else None
        }
    }, f, indent=2, ensure_ascii=False)

# 주문별 상세 저장 + Top 50
if len(per_order_df):
    per_order_df.sort_values("F1_diff(BERT-XGB)", ascending=False).head(50).to_csv(OUT_BERT_TOP50, index=False)
    per_order_df.sort_values("F1_diff(BERT-XGB)", ascending=True).head(50).to_csv(OUT_XGB_TOP50, index=False)
    per_order_df.to_csv(OUT_PER_ORDER, index=False)

print("\n===== 결과 =====")
print(summary_df)
print("\n저장됨:")
print(" - 요약 CSV:", OUT_SUMMARY_CSV)
print(" - 요약 JSON:", OUT_SUMMARY_JSON)
print(" - 주문별 F1:", OUT_PER_ORDER)
print(" - 샘플 헤더:", OUT_SAMPLES_CSV)
print(" - BERT 우세 TOP50:", OUT_BERT_TOP50)
print(" - XGB 우세 TOP50:", OUT_XGB_TOP50)


[INFO] Load master: /content/drive/MyDrive/data/instacart/master_dataset_with_roles_final.csv
[INFO] GT orders: 122,607  | rows in df_test: 1,384,617
[INFO] Load XGB orders: /content/drive/MyDrive/data/instacart/test_predictions_orders_threshold_0.32.csv
[INFO] XGB orders loaded: 131,209
[INFO] Load BERT file: /content/drive/MyDrive/data/instacart/address(in).csv
[INFO] BERT cols: ['user_id', 'eval_set', 'last_product_ids']
[INFO] BERT orders loaded: 131,184

===== 성능 요약 (Instacart F1) =====
XGB  | orders used: 122,607 / 122,607 (cov=100.00%) | F1 = 0.75182
BERT | orders used: 122,594 / 122,607 (cov=99.99%) | F1 = 0.74758

===== 결과 =====
         model  macro_F1  orders_used  coverage      AUC   Logloss
0   XGB_orders  0.751821       122607  1.000000  0.72942  0.593559
1  BERT_orders  0.747584       122594  0.999894      NaN       NaN

저장됨:
 - 요약 CSV: /content/drive/MyDrive/data/instacart/bert_xgb_comparison_summary.csv
 - 요약 JSON: /content/drive/MyDrive/data/instacart/bert_xgb_compari